### Reading the PDF and Extracting the text

In [12]:
from pypdf import PdfReader

In [13]:
pdf = []
try:
    reader = PdfReader('./PDFs/bom.pdf')
    for page in reader.pages:
         pdf.append(page.extract_text())

    print(f"Total Pages: {len(pdf)}")
    print("------------- Page 0 ------------")
    print(":::: Header ::::")
    print(pdf[0][:300]) 
    print(":::: Footer ::::")
    print(pdf[0][-300:])
    
    print("------------- Page 1 ------------")
    print(":::: Header ::::")
    print(pdf[1][:300]) 
    print(":::: Footer ::::")
    print(pdf[1][-300:])

    print("------------- Page 2 ------------")
    print(":::: Header ::::")
    print(pdf[2][:300])
    print(":::: Footer ::::")
    print(pdf[2][-300:])

    print("------------- Page 3 ------------")
    print(":::: Header ::::")
    print(pdf[3][:300])
    print(":::: Footer ::::")
    print(pdf[3][-300:])


except Exception as exc:
    # TO Catch error 
    print("Remaining")

Total Pages: 224
------------- Page 0 ------------
:::: Header ::::
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 

:::: Footer ::::
d Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
 
------------- Page 1 ------------
:::: Header ::::
      Basics of Mathematics                                                                                                                    2 
 
Section A 
A.1 Introduction to Qualitative and 
Quantitative Analysis 
Qualitative Data Analysis 
Qualitative data

### Text Cleaning

In [14]:
import re

def normalize_text(text: str) -> str:
    """ 
    Cleans extracted PDF text for better chunking and retrieval. 
    """
    # Step 1: Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # # Step 2: Remove repeated book title/header
    # text = re.sub("Basics of Mathematics\s+\d+\s*", "", text)

    # # Step 3: Remove standalone page numbers
    # text = re.sub(r"\n\s*\d+\s*\n", "\n", text)

    # Step 4: Replace multiple space with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Step 5: Replace multiple blank lines with at most two
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text

In [15]:
normalized_pdf = []
i = 0

for page in pdf:
    normalized_page = normalize_text(page)
    normalized_pdf.append(normalized_page)    

In [16]:
for i in range(0,5):
    print("---------------------------------------")
    print("------------- Original PDF ------------") 
    print(pdf[i])
    print("---------------------------------------\n")

    print("---------------------------------------")
    print("------------- Cleaned PDF ------------") 
    print(normalized_pdf[i])
    print("---------------------------------------\n")

---------------------------------------
------------- Original PDF ------------
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 
7. Simplifications 58 
8. Ratio and Proportion 69 
9. Chain rule 
74 
10. Averages 82 
11. Percentages 91 
12. Profit and Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
 
---------------------------------------

---------------------------------------
------------- Cleaned PDF ------------
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quant

### Identifying the Boilerplate and Removing it

In [23]:
from rapidfuzz import fuzz

def compute_line_similarity(pdf, n_lines=3, similarity_fn=fuzz.ratio):
    """
    Compare the first and last `n_lines` of consecutive PDF pages.

    Parameters
    ----------
    pdf : list[str]
        List containing page-wise extracted text.

    n_lines : int
        Number of header/footer candidate lines.

    similarity_fn : function
        RapidFuzz similarity function.
        Examples:
            fuzz.ratio
            fuzz.partial_ratio
            fuzz.token_sort_ratio
            fuzz.token_set_ratio
            fuzz.WRatio

    Returns
    -------
    list[dict]
        Similarity information for every consecutive page pair.
    """

    results = []

    for page_no, (current, nxt) in enumerate(zip(pdf, pdf[1:]), start=1):

        current_lines = [line.strip() for line in current.splitlines() if line.strip()]
        next_lines = [line.strip() for line in nxt.splitlines() if line.strip()]

        current_header = current_lines[:n_lines]
        next_header = next_lines[:n_lines]

        current_footer = current_lines[-n_lines:]
        next_footer = next_lines[-n_lines:]

        page_result = {
            "pages": (page_no, page_no + 1),
            "header_scores": [],
            "footer_scores": []
        }

        # -------- Header Comparison --------
        for idx, (line1, line2) in enumerate(zip(current_header, next_header), start=1):

            score = similarity_fn(line1, line2)

            page_result["header_scores"].append({
                "line": idx,
                "text1": line1,
                "text2": line2,
                "score": score
            })

        # -------- Footer Comparison --------
        for idx, (line1, line2) in enumerate(zip(current_footer, next_footer), start=1):

            score = similarity_fn(line1, line2)

            page_result["footer_scores"].append({
                "line": idx,
                "text1": line1,
                "text2": line2,
                "score": score
            })

        results.append(page_result)

    return results

compute_line_similarity(pdf, 3)

[{'pages': (1, 2),
  'header_scores': [{'line': 1,
    'text1': 'Basics of Mathematics',
    'text2': 'Basics of Mathematics                                                                                                                    2',
    'score': 26.415094339622648},
   {'line': 2,
    'text1': 'Sr. No. Topic name Page number',
    'text2': 'Section A',
    'score': 20.512820512820518},
   {'line': 3,
    'text1': '1. Qualitative and Quantitative Analysis 2',
    'text2': 'A.1 Introduction to Qualitative and',
    'score': 49.350649350649356}],
  'footer_scores': [{'line': 1,
    'text1': '19. Series 205',
    'text2': 'experience (UX)',
    'score': 34.48275862068966},
   {'line': 2,
    'text1': '20. Problems on Ages   210',
    'text2': 'Researching competitors’ website pages to identify their competitive advantages and value',
    'score': 24.347826086956527},
   {'line': 3,
    'text1': '21. Discounts – its types and Analysis 216',
    'text2': 'propositions',
    'score

In [ ]:
from rapidfuzz import fuzz

# text1 = "Basics of Mathematics 17"
# text2 = "Basics of Mathematics 18"

# score = fuzz.ratio(text1, text2)

# print(score)


# fuzz.ratio(
#     "Basics of Mathematics",
#     "Basics of Mathematics"
# )

# fuzz.ratio(
#     "Basics of Mathematics",
#     "Computer Networks"
# )

# fuzz.ratio(
#     "Page 17",
#     "Page 18"
# )

# fuzz.ratio(
#     "Chapter 4",
#     "Chapter 5"
# )

88.88888888888889

In [44]:
text = "".join(normalized_pdf)

text

"Basics of Mathematics \nSr. No. Topic name Page number \n1. Qualitative and Quantitative Analysis 2 \n2. Number and Number System, problems on Numbers 7 \n3. Fractions, Decimals and Numbers 32 \n4. Divisibility and Divisibility Test 39 \n5. HCF and LCM 44 \n6. Squares, Cubes, Square-roots & Cube-roots 51 \n7. Simplifications 58 \n8. Ratio and Proportion 69 \n9. Chain rule \n74 \n10. Averages 82 \n11. Percentages 91 \n12. Profit and Loss 101 \n13. Simple and Compound Interest 110 \n14. Heights, Distances & Basic Trigonometry 135 \n15. Time, Work and Wages 142 \n16. Time and Distances 162 \n17. Trains, Boats and Streams 176 \n18. Clock 197 \n19. Series 205 \n20. Problems on Ages 210 \n21. Discounts – its types and Analysis 216 \n \n \n  Basics of Mathematics 2 \n \nSection A \nA.1 Introduction to Qualitative and \nQuantitative Analysis \nQualitative Data Analysis \nQualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting \nqualitative data—non-numeric, c

### Chunking the PDF
:-- Using recursive chunking

In [45]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [46]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        " ",
        ".",
        ",",
        "\u200b",  # Zero-width space
        "\uff0c",  # Fullwidth comma
        "\u3001",  # Ideographic comma
        "\uff0e",  # Fullwidth full stop
        "\u3002",  # Ideographic full stop
        "",
    ],
)

all_splits = text_splitter.split_text(text)

print(f"Split PDF text into {len(all_splits)} sub-documents.")


Split PDF text into 30 sub-documents.


In [ ]:
for i, chunk in enumerate(all_splits):
    print(f"\n--- Chunk {i+1} ---")
    print(f"\n--- First half of chunk {i+1} ---")
    print(chunk[:300])
    print(f"\n--- Second half of chunk {i+1} ---")
    print(chunk[300:])
    print(f"\nLength: {len(chunk)}")

    if i == 2:
        break


--- Chunk 1 ---

--- First half of chunk 1 ---
Basics of Mathematics 
Sr. No. Topic name Page number 
1. Qualitative and Quantitative Analysis 2 
2. Number and Number System, problems on Numbers 7 
3. Fractions, Decimals and Numbers 32 
4. Divisibility and Divisibility Test 39 
5. HCF and LCM 44 
6. Squares, Cubes, Square-roots & Cube-roots 51 


--- Second half of chunk 1 ---
7. Simplifications 58 
8. Ratio and Proportion 69 
9. Chain rule 
74 
10. Averages 82 
11. Percentages 91 
12. Profit and Loss 101 
13. Simple and Compound Interest 110 
14. Heights, Distances & Basic Trigonometry 135 
15. Time, Work and Wages 142 
16. Time and Distances 162 
17. Trains, Boats and Streams 176 
18. Clock 197 
19. Series 205 
20. Problems on Ages   210 
21. Discounts – its types and Analysis 216 
 
 
       Basics of Mathematics                                                                                                                    2 
 
Section A 
A.1 Introduction to Qualitative and 
Qua

### Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(all_splits)

print(f"Each embedding has {embeddings.shape[1]} dimensions")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Each embedding has 384 dimensions


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
similarities = cosine_similarity(embeddings)
print("Similarity matrix:")
print(similarities)
# You'll see sentences 1, 2, and 4 are more similar to each other


Similarity matrix:
[[0.9999999  0.27995884 0.32016426 ... 0.5082781  0.4807489  0.4872148 ]
 [0.27995884 0.9999999  0.76331556 ... 0.1517172  0.17621809 0.12574951]
 [0.32016426 0.76331556 1.         ... 0.24193653 0.25580803 0.23576777]
 ...
 [0.5082781  0.1517172  0.24193653 ... 1.         0.9651926  0.90530217]
 [0.4807489  0.17621809 0.25580803 ... 0.9651926  1.         0.88975585]
 [0.4872148  0.12574951 0.23576777 ... 0.90530217 0.88975585 1.        ]]


## Storing into the Database

In [ ]:
import faiss

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

In [ ]:
print(index.ntotal)

66


In [ ]:
# query = "What is qualitative data analysis?"
query = "What is Qualitative Analysis?"

query_embedding = model.encode([query])


## Searching into Database

In [ ]:
k = 3

distances, indices = index.search(
    query_embedding,
    k
)

In [ ]:
retrived_text = []
for idx in indices[0]:
    retrived_text.append(all_splits[idx])

retrived_text

["Section A \nA.1 Introduction to Qualitative and \nQuantitative Analysis \nQualitative Data Analysis \nQualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting \nqualitative data—non-numeric, conceptual information and user feedback—to capture \nthemes and patterns, answer research questions, and identify actions to take to improve your \nproduct or website. \nHere are five methods of qualitative data analysis to help you make sense of the data you've \ncollected through customer interviews, surveys, and feedback: \n1. Content analysis \n2. Thematic analysis \n3. Narrative analysis \n4. Grounded theory analysis \n5. Discourse analysis \n1. Content analysis \nContent analysis is a research method that examines and quantifies the presence of certain \nwords, subjects, and concepts in text, image, video, or audio messages. The \nmethod transforms qualitative input into quantitative data to help you make reliable",
 'In a business context, the method is pr

## Response Generation using LLM
---

### Cell 1
#### Responsibility : - Configure LLM Model

In [ ]:
from google import genai

In [ ]:
# Client Configuration
API_KEY = 'AQ.Ab8RN6LdZl8pxdRhnCTAMsjCvQuzFv1CRJt3JGnqdkmhUR-GhQ'

client = genai.Client(api_key=API_KEY)

### Cell 2
#### Responsibility : - Prompt Builder

In [ ]:
context = "\n\n".join(retrived_text)

prompt = f"""
You are an AI assistant.

Answer ONLY using the provided context.

If the answer is not available,
say "I don't know."

Context:
{context}

Query:
{query}
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents= prompt
)

In [ ]:
print("response")
print(response.text)

response
Qualitative data analysis (QDA) is the process of organizing, analyzing, and interpreting qualitative data—non-numeric, conceptual information and user feedback—to capture themes and patterns, answer research questions, and identify actions to take to improve your product or website.
